## This notebook compiles crispresso results.

Kexin Dong

Nov 25, 2025

## Load library, config file, separate ABE and CBE

In [ ]:
import numpy as np 
import pandas as pd
import os
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']

LIB = pd.read_csv('.../FINAL_focused_library.csv')
config = pd.read_csv('.../CONFIG_BALL_VALIDATION_SCREEN.txt', sep=' ')
#filepath for crispresso data
fp = '...crispresso/original_unmerged'

## Define key functions

### `crispresso_compiler`

Reads each per-sample aggregated CRISPResso CSV (output of STEP6) and turns it into a tidy per-guide editing table. For every `Guide_ID` in a sample, it pulls counts from two amplicons — `Reference` (WT) and `HDR` (edited) — and computes:

- **Read accounting:** `Reads_in_input`, `Reads_lowqual`, `Reads_aligned_all_amplicons`, `Reads_aligned_WT`, `Reads_aligned_HDR`, `Reads_unaligned`
- **Editing outcomes:** `correct_edit` (clean HDR), `target_base_edit` (HDR including bystanders), `WT` (unmodified reference)
- **Byproducts split into types:** `byproduct_INDEL`, `byproduct_sub`, `byproduct_ambiguous`, and the total `byproduct_all` (unaligned reads are folded in here)
- **Percent versions** of all of the above, normalised by `Reads_aligned_all_amplicons`

Returns a `dict` keyed by the user-friendly `sample_id` you pass in (so you can call samples `'d5-1'` instead of long `D25-xxxxx` identifiers).


In [3]:
def crispresso_compiler(samp, sample_id, fp):
    """ 
    Takes in list of sample names (see cell above)
    Returns compiled dictionary of dataframes containing editing information for each sensor
    - key for each is the sample_id name
    
    Includes:
    1. corr_perc = pure correct editing
    2. target_base_edit_perc = target base editing perc (including edits with bystander editing)
    3. wt_perc
    4. byproduct information (indels, substitutions, ambiguous)
    """

    df_edits = []
    for k in samp: 
        concated = pd.read_csv(f'{fp}/{k}_crispresso_aggregated.csv')
        concated = concated.fillna(0)

        #somehow wasn't able to find the initial set of code that I used for this...

        #go through all of the samples and do it step by step
        sample_ids = []
        #sample_num = []
        rii = []
        raaa = []
        r_lowqual = []
        ra_hdr = []
        ra_wt = []
        r_unaligned = []
        no_edit = []
        correct = []
        target_base_editing = []
        byproduct_all = []
        byproduct_indel = []
        byproduct_sub = []
        byproduct_ambig = []
        for i in np.unique(concated['Guide_ID']):
            sample_ids.append(i)

            subset = concated[concated['Guide_ID']==i]

            wt = subset[subset['Amplicon']=='Reference']
            edit = subset[subset['Amplicon']=='HDR']

            #sample_num.append(wt['sample_num'].values[0])
            rii1 = wt['Reads_in_input'].values[0]
            r1 = wt['Reads_aligned_all_amplicons'].values[0]
            rii.append(rii1)
            raaa.append(r1)
            r_lowqual.append(rii1-r1)

            r2 = edit['Reads_aligned'].values[0]
            r3 = wt['Reads_aligned'].values[0]
            ra_hdr.append(r2)
            ra_wt.append(r3)
            r_unaligned.append(r1 - (r2+r3))

            no_edit.append(wt['Unmodified'].values[0])
            correct.append(edit['Unmodified'].values[0])
            target_base_editing.append(edit['Modified'].values[0] + edit['Unmodified'].values[0])

            byprod_all = wt['Modified'].values[0] + edit['Modified'].values[0] + (r1 - (r2+r3)) #add unaligned reads
            sub_all = wt['Only Substitutions'].values[0] + edit['Only Substitutions'].values[0]
            indel_all = wt['Only Deletions'].values[0] + wt['Only Insertions'].values[0] + wt['Insertions and Deletions'].values[0] + edit['Only Deletions'].values[0] + edit['Only Insertions'].values[0] + edit['Insertions and Deletions'].values[0]
            ambig_all = byprod_all - sub_all - indel_all

            byproduct_all.append(byprod_all)
            byproduct_indel.append(indel_all)
            byproduct_sub.append(sub_all)
            byproduct_ambig.append(ambig_all)


        cols = ["Guide_ID", "Reads_in_input","Reads_lowqual", "Reads_aligned_all_amplicons","Reads_aligned_WT", "Reads_aligned_HDR", "Reads_unaligned", "WT","correct_edit", "target_base_edit", "byproduct_all","byproduct_INDEL","byproduct_sub","byproduct_ambiguous"]
        col_vals = [ sample_ids, rii, r_lowqual, raaa, ra_wt, ra_hdr, r_unaligned, no_edit, correct, target_base_editing, byproduct_all, byproduct_indel, byproduct_sub, byproduct_ambig]

        out = pd.DataFrame(dict(zip(cols, col_vals)))
        out['corr_perc'] = 100*(out['correct_edit']/out['Reads_aligned_all_amplicons'])
        out['target_base_edit_perc'] = 100*(out['target_base_edit']/out['Reads_aligned_all_amplicons'])
        out['WT_perc'] = 100*(out['WT']/out['Reads_aligned_all_amplicons'])
        out['byproduct_all_perc'] =  100*(out['byproduct_all']/out['Reads_aligned_all_amplicons'])
        out['byproduct_INDEL_perc'] =  100*(out['byproduct_INDEL']/out['Reads_aligned_all_amplicons'])
        out['byproduct_sub_perc'] =  100*(out['byproduct_sub']/out['Reads_aligned_all_amplicons'])
        out['byproduct_ambiguous_perc'] =100*(out['byproduct_ambiguous']/out['Reads_aligned_all_amplicons'])
        out = out.fillna(0)
        out = out.sort_index()
        df_edits.append(out)

    edit_dict = dict(zip(sample_id, df_edits))

    return edit_dict

### `MLE_merge_auto`

Combines biological/technical replicates of the same condition by **summing the raw count columns** and then recomputing the percentages from the merged totals. This is the maximum-likelihood-style merge: it treats each replicate's reads as independent observations of the same underlying editing rate, so a sample with 10× more reads carries 10× the weight.

Inputs:

- `edit_dict` — output of `crispresso_compiler`
- `samps_to_merge` — list of groups, e.g. `[['bm1','bm2','bm3'], ['d15-rep1','d15-rep2']]`
- `new_name` — the merged label for each group (same length as `samps_to_merge`)

Works for groups of any size ≥ 1, and asserts that `Guide_ID` order matches across replicates before adding.


In [ ]:
def MLE_merge_auto(edit_dict, samps_to_merge, new_name):
    """
    Combine replicates to generate best estimate of sensor editing.
    Automatically handles any number of replicates per group (≥1).
    """
    cols_to_add = [
        'Reads_in_input', 'Reads_lowqual', 'Reads_aligned_all_amplicons',
        'Reads_aligned_WT', 'Reads_aligned_HDR', 'Reads_unaligned', 'WT',
        'correct_edit', 'target_base_edit', 'byproduct_all',
        'byproduct_INDEL', 'byproduct_sub', 'byproduct_ambiguous',
    ]
    
    comb_holder = []

    for group in samps_to_merge:
        dfs = [edit_dict[sample] for sample in group]
        guide_check = [list(df['Guide_ID']) for df in dfs]
        assert all(g == guide_check[0] for g in guide_check), 'Guide_ID dont match'

        merged = dfs[0][cols_to_add].copy()
        for df in dfs[1:]:
            merged += df[cols_to_add]

        merged['Guide_ID'] = dfs[0]['Guide_ID']
        col_order = ['Guide_ID'] + cols_to_add
        out = merged[col_order]

        out['corr_perc'] = 100 * (out['correct_edit'] / out['Reads_aligned_all_amplicons'])
        out['target_base_edit_perc'] = 100 * (out['target_base_edit'] / out['Reads_aligned_all_amplicons'])
        out['WT_perc'] = 100 * (out['WT'] / out['Reads_aligned_all_amplicons'])
        out['byproduct_all_perc'] = 100 * (out['byproduct_all'] / out['Reads_aligned_all_amplicons'])
        out['byproduct_INDEL_perc'] = 100 * (out['byproduct_INDEL'] / out['Reads_aligned_all_amplicons'])
        out['byproduct_sub_perc'] = 100 * (out['byproduct_sub'] / out['Reads_aligned_all_amplicons'])
        out['byproduct_ambiguous_perc'] = 100 * (out['byproduct_ambiguous'] / out['Reads_aligned_all_amplicons'])
        out = out.fillna(0)
        out = out.sort_index()   # keep original row order
        comb_holder.append(out)

    return dict(zip(new_name, comb_holder))

### `merge_NGS_replicates`

Thin wrapper over `MLE_merge_auto` for the **lane-merge** case: when one biological sample was sequenced across two NGS lanes, the sample list looks like `['d5-1','d5-2','d15-rep1-1','d15-rep1-2', ...]`. This function pairs every two consecutive entries (`-1` / `-2`), merges them, and strips the trailing `-N` to produce a clean per-sample name (e.g. `'d5'`, `'d15-rep1'`).

Use this **before** doing biological-replicate merging with `MLE_merge_auto`.


In [ ]:
def merge_NGS_replicates(sample_ids, edit_dict): 
    samps_to_merge = [
        [sample_ids[i], sample_ids[i+1]]
        for i in range(0, len(sample_ids), 2)
    ]
    new_sample_ids = list(dict.fromkeys(x.rsplit("-", 1)[0] for x in sample_ids))
    comb_dict = MLE_merge_auto(edit_dict, samps_to_merge, new_sample_ids)
    return comb_dict

### `MLE_merge` (legacy, fixed at 3 replicates)

Older version of `MLE_merge_auto` that **only handles groups of exactly 3 replicates**. Kept here for reproducibility with earlier analyses. For new work, prefer `MLE_merge_auto`.


In [ ]:
def MLE_merge(edit_dict, samps_to_merge, new_name):
    """ 
    combine replicates to generate best estimate of sensor editing
    only works for combinations of 3 replicates (could be modified)
    """
    cols_to_add = ['Reads_in_input', 'Reads_lowqual',
       'Reads_aligned_all_amplicons', 'Reads_aligned_WT', 'Reads_aligned_HDR',
       'Reads_unaligned', 'WT', 'correct_edit', 'target_base_edit',
       'byproduct_all', 'byproduct_INDEL', 'byproduct_sub',
       'byproduct_ambiguous',]
    
    comb_holder = []
    for k in samps_to_merge:
        a = edit_dict[k[0]]
        b = edit_dict[k[1]]
        c = edit_dict[k[2]]

        #check that guide indeces match
        assert list(a['Guide_ID'])==list(b['Guide_ID'])==list(c ['Guide_ID']), 'indexes dont match'

        a2 = a[cols_to_add]
        b2 = b[cols_to_add]
        c2 = c[cols_to_add]

        comb = a2+b2+c2

        comb['Guide_ID'] = a['Guide_ID']

        #and reorganize columns
        col_order = ['Guide_ID'] + cols_to_add
        out = comb[col_order]

        #and then calcualte percentages
        out['corr_perc'] = 100*(out['correct_edit']/out['Reads_aligned_all_amplicons'])
        out['target_base_edit_perc'] = 100*(out['target_base_edit']/out['Reads_aligned_all_amplicons'])
        out['WT_perc'] = 100*(out['WT']/out['Reads_aligned_all_amplicons'])
        out['byproduct_all_perc'] =  100*(out['byproduct_all']/out['Reads_aligned_all_amplicons'])
        out['byproduct_INDEL_perc'] =  100*(out['byproduct_INDEL']/out['Reads_aligned_all_amplicons'])
        out['byproduct_sub_perc'] =  100*(out['byproduct_sub']/out['Reads_aligned_all_amplicons'])
        out['byproduct_ambiguous_perc'] =100*(out['byproduct_ambiguous']/out['Reads_aligned_all_amplicons'])
        out = out.fillna(0)

        comb_holder.append(out)

    out_dict = dict(zip(new_name, comb_holder))

    return out_dict

## Sample-name dictionary

Each lane of each biological sample comes back from the sequencing core with a cryptic `D25-xxxxx-N-7331E` ID. Here we:

1. List the raw IDs in a consistent order (library → input → time-course → tissues),
2. Append the suffix `_guide_split_BARCODES` so they match the folder names that STEP4 produced,
3. Build a parallel list of **human-readable sample IDs** (`lib-1`, `input-1`, `d5-1`, `d15-rep1-1`, `spleen1-1`, …) that we'll use from this point on.

Finally we run `crispresso_compiler` to load all aggregated CSVs into `edit_dict_ABE_BC`, keyed by the friendly names.


In [ ]:
samp_ABE_BC = [
    'D25-13337-1-7331E','D25-13337-2-7331E', #library

    'D25-13245-1-7331E','D25-13245-2-7331E', #input

    'D25-13251-1-7331E','D25-13251-2-7331E', #in vitro d5

    'D25-13317-1-7331E','D25-13317-2-7331E', #in vitro d15
    'D25-13788-1-7331E','D25-13788-2-7331E',
    'D25-13789-1-7331E','D25-13789-2-7331E',
    'D25-13790-1-7331E','D25-13790-2-7331E',
    'D25-13321-1-7331E','D25-13321-2-7331E',

    'D25-13278-1-7331E','D25-13278-2-7331E', #spleen1
    'D25-13279-1-7331E','D25-13279-2-7331E', #bm1
    'D25-13280-1-7331E','D25-13280-2-7331E', #men1
    'D25-13281-1-7331E','D25-13281-2-7331E',
    'D25-13282-1-7331E','D25-13282-2-7331E',
    'D25-13283-1-7331E','D25-13283-2-7331E',
    'D25-13284-1-7331E','D25-13284-2-7331E',
    'D25-13285-1-7331E','D25-13285-2-7331E',
    'D25-13286-1-7331E','D25-13286-2-7331E',
    'D25-13287-1-7331E','D25-13287-2-7331E',
    'D25-13288-1-7331E','D25-13288-2-7331E',
    'D25-13289-1-7331E','D25-13289-2-7331E',
    'D25-13290-1-7331E','D25-13290-2-7331E',
    'D25-13291-1-7331E','D25-13291-2-7331E',
    'D25-13292-1-7331E','D25-13292-2-7331E',
]
samp_ABE_BC = [f'{x}_guide_split_BARCODES' for x in samp_ABE_BC]
id_abe_bc_d15 = []
for y in range(5):
    for z in range(2):
        id_abe_bc_d15.append(f'd15-rep{y+1}-{z+1}')

id_in_vivo = []
for y in range(5):
    for x in ['spleen','bm','men']:
        for z in range(2):
            id_in_vivo.append(f'{x}{y+1}-{z+1}')
            
samp_id_ABE_BC = [
    'lib-1','lib-2',
    'input-1','input-2',
    'd5-1','d5-2',
] + id_abe_bc_d15 + id_in_vivo

edit_dict_ABE_BC = crispresso_compiler(samp_ABE_BC, samp_id_ABE_BC, fp)

Run the lane merger. After this, sample names lose their `-1` / `-2` lane suffix (e.g. `'d5-1','d5-2'` → `'d5'`), but biological replicates (`bm1`, `bm2`, …) are still separate — we'll merge those in the MLE step below.


In [ ]:
ABE_BC = merge_NGS_replicates(samp_id_ABE_BC, edit_dict_ABE_BC)

### Save lane-merged tables

Write each lane-merged sample to its own CSV under `compact_lane_merged_unfiltered/<screen_name>/`. This is the checkpoint to come back to next time — you can skip the cluster outputs and reload directly from here.

> **Heads-up:** update `fp` to a path on your machine before running. The path here is hard-coded to my (KD) local repo.


In [ ]:
fp = '/Users/kexindong/Documents/GitHub/PhD-FSR-MH-Lab/07_B-ALL_resubmission_20250819/validation_screen_analysis/crispresso_data/compact_lane_merged_unfiltered'
comb_dict = [ABE_BC]
name_list = ['ABE_BC']

for i in range(len(comb_dict)):
    single_dict = comb_dict[i]
    name = name_list[i]
    os.makedirs(os.path.join(fp, name), exist_ok=True)
    for sample_name, df in single_dict.items():
        filename = os.path.join(fp, name, f'{sample_name}_compact_unfiltered.csv')
        # print(filename)
        df.to_csv(filename, index=False)

### Reload from disk

If you've already run the compile + lane-merge steps once, start here on subsequent sessions. Reads every `*_compact_unfiltered.csv` back into `loaded_dict[screen_name][sample_name]`.


In [ ]:
# next time to use them:
fp = '/Users/kexindong/Documents/GitHub/PhD-FSR-MH-Lab/07_B-ALL_resubmission_20250819/validation_screen_analysis/crispresso_data/compact_lane_merged_unfiltered'

name_list = ['ABE_BC']

loaded_dict = {}   # this will hold everything

for name in name_list:
    folder_path = os.path.join(fp, name)
    sample_files = sorted(os.listdir(folder_path))   # alphabetical order
    
    # dictionary for this screen
    screen_dict = {}

    for fname in sample_files:
        if fname.endswith('.csv'):
            sample_name = fname.replace('_compact_unfiltered.csv', '')
            fpath = os.path.join(folder_path, fname)
            screen_dict[sample_name] = pd.read_csv(fpath)

    loaded_dict[name] = screen_dict

## Biological-replicate merging (MLE)

Define which lane-merged samples belong to the same biological condition. Each list inside `*_samps_to_merge` becomes one row of merged output, labelled by the corresponding entry in `*_new_name`.

Note that the **CBE_BC** group has fewer replicates for some tissues (e.g. `['bm1','bm2','bm4']`) — some replicates were excluded due to QC issues. `MLE_merge_auto` handles uneven group sizes automatically.


In [72]:
ABE_EPO_samps_to_merge = ABE_BC_samps_to_merge = [['bm1', 'bm2', 'bm3', 'bm4', 'bm5'], 
['d15-rep1', 'd15-rep2', 'd15-rep3','d15-rep4', 'd15-rep5'], 
['d5'],
['input'],
['lib'],
['men1', 'men2', 'men3','men4', 'men5'], 
['spleen1', 'spleen2', 'spleen3', 'spleen4', 'spleen5']]
ABE_EPO_new_name = ABE_BC_new_name = ['bm', 'd15', 'd5','input','lib','men', 'spleen']

CBE_BC_samps_to_merge = [['bm1', 'bm2', 'bm4'], 
['d15-rep1', 'd15-rep2', 'd15-rep3','d15-rep4', 'd15-rep5'], 
['d5-rep1', 'd5-rep2', 'd5-rep3','d5-rep4', 'd5-rep5'],
['input-rep1','input-rep2','input-rep3','input-rep4','input-rep5'],
['lib'],
['men1', 'men2', 'men4'], 
['spleen1', 'spleen2', 'spleen4']]
CBE_BC_new_name = ['bm', 'd15', 'd5','input','lib','men', 'spleen']

CBE_EPO_samps_to_merge = [['bm1', 'bm2', 'bm3', 'bm4', 'bm5'], 
['d15-rep1', 'd15-rep2', 'd15-rep3','d15-rep4', 'd15-rep5'], 
['d5-rep1', 'd5-rep2', 'd5-rep3','d5-rep4', 'd5-rep5'],
['input-rep1','input-rep2','input-rep3','input-rep4','input-rep5'],
['lib'],
['men1', 'men2', 'men3','men4', 'men5'], 
['spleen1', 'spleen2', 'spleen3', 'spleen4', 'spleen5']]
CBE_EPO_new_name = ['bm', 'd15', 'd5','input','lib','men', 'spleen']

Apply `MLE_merge_auto` to each of the four sub-screens (ABE × BC/EPO, CBE × BC/EPO) to collapse biological replicates into one per condition.


In [ ]:
MLE_ABE_BC = MLE_merge_auto(loaded_dict['ABE_BC'], ABE_BC_samps_to_merge, ABE_BC_new_name)
MLE_ABE_EPO = MLE_merge_auto(loaded_dict['ABE_EPO'], ABE_EPO_samps_to_merge, ABE_EPO_new_name)
MLE_CBE_BC = MLE_merge_auto(loaded_dict['CBE_BC'], CBE_BC_samps_to_merge, CBE_BC_new_name)
MLE_CBE_EPO = MLE_merge_auto(loaded_dict['CBE_EPO'], CBE_EPO_samps_to_merge, CBE_EPO_new_name)

### Save MLE-merged tables

Final output: one CSV per condition per screen, under `MLE/<screen_name>/<condition>.csv`. These are the files used downstream (e.g. for LFC calculation in STEP8).

> **Heads-up:** update `fp` to a path on your machine before running.


In [74]:
fp = '/Users/kexindong/Documents/GitHub/PhD-FSR-MH-Lab/07_B-ALL_resubmission_20250819/validation_screen_analysis/MLE'
comb_dict = [MLE_ABE_BC, MLE_ABE_EPO, MLE_CBE_BC, MLE_CBE_EPO]
name_list = ['ABE_BC', 'ABE_EPO', 'CBE_BC','CBE_EPO']

for i in range(len(comb_dict)):
    single_dict = comb_dict[i]
    name = name_list[i]
    os.makedirs(os.path.join(fp, name), exist_ok=True)
    for sample_name, df in single_dict.items():
        filename = os.path.join(fp, name, f'{sample_name}.csv')
        # print(filename)
        df.to_csv(filename, index=False)